In [1]:
URDF_FILE = r"C:\\Chethan\\Technologies\\Robotics\\Major_Project\\mirs\\src\\mirs_description\\urdf\\robot.xacro"

In [30]:
import numpy as np
import math 
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sympy import *

# init_session()

def dh_matrix(a, alpha, d, theta):
    """
    Compute the standard DH transformation matrix using NumPy.
    
    Parameters:
        a / r : link length
        alpha : link twist (in radians)
        d     : link offset
        theta : joint angle (in radians)

    Returns:
        4x4 NumPy array representing the transformation matrix
    """
    return Matrix([
        [cos(theta), -sin(theta)*math.cos(alpha),  sin(theta)*math.sin(alpha), a*cos(theta)],
        [sin(theta),  cos(theta)*math.cos(alpha), -cos(theta)*math.sin(alpha), a*sin(theta)],
        [0,              math.sin(alpha),                math.cos(alpha),               d],
        [0,              0,                            0,                           1]
    ])

def dh_matrix_np(a, alpha, d, theta):
    """
    Compute the standard DH transformation matrix using NumPy.
    
    Parameters:
        a / r : link length
        alpha : link twist (in radians)
        d     : link offset
        theta : joint angle (in radians)

    Returns:
        4x4 NumPy array representing the transformation matrix
    """
    return np.array([
        [math.cos(theta), -math.sin(theta)*math.cos(alpha),  math.sin(theta)*math.sin(alpha), a*math.cos(theta)],
        [math.sin(theta),  math.cos(theta)*math.cos(alpha), -math.cos(theta)*math.sin(alpha), a*math.sin(theta)],
        [0,              math.sin(alpha),                math.cos(alpha),               d],
        [0,              0,                            0,                           1]
    ])


In [31]:
masses = np.array([2,3.42,1.26, 0.8,0.8,0.35])
COM = np.array([[0, -0.02, 0],
        [0.13, 0, 0.1157],
        [0.05, 0, 0.0238],
        [0, 0, 0.01],
        [0, 0, 0.01],
        [0, 0, -0.02]])


# theta, a, d, alpha
dh_params = np.array([
                    [0,	0,	0.1519,	math.pi/2],		
                    [0,	-0.24365,	0,	0],
                    [0,	-0.21325,	0,	0],
                    [0,	0,	0.11235,	math.pi/2],
                    [0,	0,	0.08535,	-math.pi/2],
                    [0,	0,	0.0819,	0]])
d_1 = dh_params[0][2]
d_2 = dh_params[1][2]
d_3 = dh_params[2][2]
d_4 = dh_params[3][2]
d_5 = dh_params[4][2]
d_6 = dh_params[5][2]
a_1 = dh_params[0][1]
a_2 = dh_params[1][1]
a_3 = dh_params[2][1]
a_4 = dh_params[3][1]
a_5 = dh_params[4][1]
a_6 = dh_params[5][1]

In [ ]:
T_goal @ dh_matrix_np(a, alpha, d, theta)

In [45]:
T_goal = np.eye(4)

for i in range(6):
    theta, a, d, alpha = dh_params[i]
    T_goal = T_goal @ dh_matrix_np(a, alpha, d, theta)
T_goal = np.round(T_goal,4)

print(T_goal)

[[ 1.      0.      0.     -0.4569]
 [ 0.      0.     -1.     -0.1943]
 [ 0.      1.      0.      0.0666]
 [ 0.      0.      0.      1.    ]]


In [46]:
R_06 = T_goal[:3,:3]
P_06 = T_goal[:3,3]
Z_6 = T_goal[:3,2]

W = P_06 - d_6*Z_6
W_x,W_y,W_z = W

W

array([-0.4569, -0.1124,  0.0666])

In [47]:
theta_1 = np.arctan2(W_y,W_x)
r = np.sqrt( W_x**2 + W_y**2)
s = W_z-d_1

theta_1,r,s

In [51]:
theta_3 = np.arccos(r**2+ s**2 - a_2**2 - a_3**2,  (2*a_2*a_3))

TypeError: return arrays must be of ArrayType

In [54]:
r**2+ s**2 - a_2**2 - a_3**2,  (2*a_2*a_3)

In [7]:


theta_21 = np.arccos(((a_2+a_3*np.cos(theta_3))*r + a_3*np.sin(theta_3)*s)/(r**2 + s**2))
theta_22 = np.arcsin(((a_2+a_3*np.cos(theta_3))*s - a_3*np.sin(theta_3)*r)/(r**2 + s**2))






R_031 = np.eye(3)
R_032 = np.eye(3)

theta_s1 = [theta_1,theta_21,theta_3]
theta_s2 = [theta_1,theta_22,theta_3]

for i,(a, alpha, d, _) in enumerate(dh_params[:3]):
    print(i)
    A = dh_matrix_np(a, alpha, d, theta_s1[i])
    R_031 = R_031 @ A[:3,:3]
for i,(a, alpha, d, _) in enumerate(dh_params[:3]):
    A = dh_matrix_np(a, alpha, d, theta_s2[i])
    R_032 = R_032 @ A[:3,:3]
   
R_361 = R_031.T @ R_06
R_362 = R_032.T @ R_06
theta51 = atan2(sqrt(R_361[0,2]**2 + R_361[1,2]**2), R_361[2,2])
theta52 = atan2(sqrt(R_362[0,2]**2 + R_362[1,2]**2), R_362[2,2])
theta41 = atan2(R_361[1,2], R_361[0,2])
theta42 = atan2(R_362[1,2], R_362[0,2])
theta61 = atan2(R_361[2,1], -R_361[2,0])
theta62 = atan2(R_362[2,1], -R_362[2,0])

0
1
2


C:\Users\cheth\AppData\Local\Temp\ipykernel_16592\1725399544.py:25: RuntimeWarning: invalid value encountered in arccos
  theta_3 = np.arccos((r**2+ s**2 - a_2**2 - a_3**2)/ (2*a_2*a_3))


In [200]:
# # Plot coordinate frame at a given transformation matrix
# def plot_frame(ax, T, length=0.05):
#     origin = T[:3, 3]
#     x_axis = T[:3, 0] * length
#     y_axis = T[:3, 1] * length
#     z_axis = T[:3, 2] * length

#     ax.quiver(*origin, *x_axis, color='r')
#     ax.quiver(*origin, *y_axis, color='g')
#     ax.quiver(*origin, *z_axis, color='b')


# # Forward kinematics
# T = np.eye(4)
# points = [T[:3, 3]]
# transforms = [T.copy()]

# for a, alpha, d, theta in dh_params:
#     A = dh_matrix(a, alpha, d, theta)
#     T = T @ A
#     points.append(T[:3, 3])
#     transforms.append(T.copy())

# points = np.array(points)

# # Plot
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# ax.plot(points[:, 0], points[:, 1], points[:, 2], '-o', linewidth=3, markersize=6, color='black')

# # Plot coordinate frames at each joint
# for T in transforms:
#     plot_frame(ax, T, length=0.001)

# # Styling
# ax.set_xlabel('X axis')
# ax.set_ylabel('Y axis')
# ax.set_zlabel('Z axis')
# ax.set_title('Robot Arm with DH Frames')
# ax.grid(True)
# ax.set_box_aspect([1, 1, 1])
# ax.view_init(elev=30, azim=45)

# plt.show()


In [10]:
from sympy import symbols, Matrix, cos, sin, pi, simplify, atan2, sqrt

# Step 1: Define symbolic joint angles
theta = symbols('θ1 θ2 θ3 θ4 θ5 θ6')

# Step 2: Define DH Parameters (theta, a, d, alpha)
DH_params = [
    (theta[0], 0,        0.1519,   pi/2),
    (theta[1], -0.24365, 0,        0),
    (theta[2], -0.21325, 0,        0),
    (theta[3], 0,        0.11235,  pi/2),
    (theta[4], 0,        0.08535, -pi/2),
    (theta[5], 0,        0.0819,   0)
]


In [11]:
# Step 3: Define the DH Transformation function
def dh_transform(theta, a, d, alpha):
    return Matrix([
        [cos(theta), -sin(theta)*cos(alpha),  sin(theta)*sin(alpha), a*cos(theta)],
        [sin(theta),  cos(theta)*cos(alpha), -cos(theta)*sin(alpha), a*sin(theta)],
        [0,           sin(alpha),             cos(alpha),            d],
        [0,           0,                      0,                     1]
    ])

# Step 4: Compute T_06 using forward kinematics
T = Matrix.eye(4)
for i in range(6):
    T = simplify(T * dh_transform(*DH_params[i]))
T06 = T

In [15]:
# Step 5: Compute Wrist Center
z_hat = Matrix([0, 0, 1])
P_wc = simplify(T06[:3, 3] - 0.0819 * T06[:3, :3] @ z_hat)
P_wc

⎡-(0.24365⋅cos(θ₂) + 0.21325⋅cos(θ₂ + θ₃))⋅cos(θ₁) + 0.11235⋅sin(θ₁) + 0.08535⋅sin(θ₂ + θ₃ + θ₄)⋅cos(θ₁)⎤
⎢                                                                                                       ⎥
⎢-(0.24365⋅cos(θ₂) + 0.21325⋅cos(θ₂ + θ₃))⋅sin(θ₁) + 0.08535⋅sin(θ₁)⋅sin(θ₂ + θ₃ + θ₄) - 0.11235⋅cos(θ₁)⎥
⎢                                                                                                       ⎥
⎣             -0.24365⋅sin(θ₂) - 0.21325⋅sin(θ₂ + θ₃) - 0.08535⋅cos(θ₂ + θ₃ + θ₄) + 0.1519              ⎦

In [242]:
# Step 6: Compute T03
T03 = Matrix.eye(4)
for i in range(3):
    T03 = simplify(T03 * dh_transform(*DH_params[i]))
R03 = T03[:3, :3]

# Step 7: Compute R36 = R03.T * R06
R06 = T06[:3, :3]
R36 = simplify(R03.T * R06)

In [247]:
# Step 8: Wrist angles (θ4, θ5, θ6) from R36
# These are symbolic expressions depending on θ1-θ3 values


θ6

In [ ]:
f








# Results
print("Forward Kinematics T06:")
print(T06)
print("\nWrist Center:")
print(P_wc)
print("\nWrist Orientation R36:")
print(R36)
print("\nSolved Wrist Angles:")
print(f"θ4: {θ4}, θ5: {θ5}, θ6: {θ6}")

In [5]:



T = []
n = 6

for i in range(n):
    T.append(get_transform_mat_from_dh(dh_params[i][0],dh_params[i][1],dh_params[i][2],theta[i]))


x,y,z,theta,beta,gamma = symbols('x y z theta beta gamma')
Ps = Matrix([[x],[y],[z],[1]])

def round_expr(expr_list, num_digits):
    T = []
    for expr in expr_list:
        T.append(expr.xreplace({n : round(n, num_digits) for n in expr.atoms(Number)}))
    return T
T = round_expr(T,4)


f_1,f_2,f_3 = symbols('f1:4')
fs = Matrix([[f_1],[f_2],[f_3],[1]])

f = T[2]*T[3][:,[3]]
f1 = f[0,0]
f2 = f[1,0]
f3 = f[2,0]


g_1,g_2,g_3 = symbols('g1:4')
gs = Matrix([[g_1],[g_2],[g_3],[1]])

g = T[1]*fs
g1 = g[0,0]
g2 = g[1,0]
g3 = g[2,0]


P = simplify(T[0]*gs)
r = simplify(P[0,0]**2 + P[1,0]**2 + P[2,0]**2)
z = P[2,0]

IPython console for SymPy 1.13.1 (Python 3.12.6-64-bit) (ground types: python)

These commands were executed:
>>> from sympy import *
>>> x, y, z, t = symbols('x y z t')
>>> k, m, n = symbols('k m n', integer=True)
>>> f, g, h = symbols('f g h', cls=Function)
>>> init_printing()

Documentation can be found at https://docs.sympy.org/1.13.1/



IndexError: invalid index to scalar variable.

In [40]:
# Goal Pose
T_06 = np.eye(4)
T_06

array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]])

NameError: name 'symbols' is not defined

In [5]:
# D-H Parameters (theta is variable)
d1, a2, a3, d4, d6 = 0.1, 0.3, 0.2, 0.4, 0.1

def rot_z(theta):
    return np.array([
        [cos(theta), -sin(theta), 0],
        [sin(theta), cos(theta), 0],
        [0, 0, 1]
    ])

def rot_y(theta):
    return np.array([
        [cos(theta), 0, sin(theta)],
        [0, 1, 0],
        [-sin(theta), 0, cos(theta)]
    ])

def rot_x(theta):
    return np.array([
        [1, 0, 0],
        [0, cos(theta), -sin(theta)],
        [0, sin(theta), cos(theta)]
    ])

def analytical_ik(T_06):
    P_06 = T_06[:3, 3]
    R_06 = T_06[:3, :3]

    # Step 1: Wrist center
    wx = P_06[0] - d6 * R_06[0, 2]
    wy = P_06[1] - d6 * R_06[1, 2]
    wz = P_06[2] - d6 * R_06[2, 2]

    # Step 2: Theta1
    theta1 = atan2(wy, wx)

    # Step 3: Theta2 and Theta3 (position IK using triangle)
    r = sqrt(wx**2 + wy**2)
    s = wz - d1

    D = (r**2 + s**2 - a2**2 - a3**2) / (2 * a2 * a3)
    if abs(D) > 1:
        raise ValueError("Target unreachable: D > 1")

    theta3 = atan2(-sqrt(1 - D**2), D) # elbow-down

    phi1 = atan2(s, r)
    phi2 = atan2(a3 * sin(theta3), a2 + a3 * cos(theta3))
    theta2 = phi1 - phi2

    # Step 4: Compute R_03
    R_01 = rot_z(theta1) @ rot_x(pi/2)
    R_12 = rot_z(theta2) @ rot_x(0)
    R_23 = rot_z(theta3) @ rot_x(-pi/2)
    R_03 = R_01 @ R_12 @ R_23

    # Step 5: Compute R_36
    R_36 = R_03.T @ R_06

    # Step 6: Extract theta4, theta5, theta6 from R_36
    theta5 = atan2(sqrt(R_36[0, 2]**2 + R_36[1, 2]**2), R_36[2, 2])
    if abs(sin(theta5)) < 1e-6:
        theta4 = 0
        theta6 = atan2(-R_36[1, 0], R_36[0, 0])
    else:
        theta4 = atan2(R_36[1, 2], R_36[0, 2])
        theta6 = atan2(R_36[2, 1], -R_36[2, 0])

    return np.array([theta1, theta2, theta3, theta4, theta5, theta6])

# Example usage
if __name__ == "__main__":
    # Desired pose
    T_06 = np.eye(4)
    T_06[:3, 3] = [0.3, 0.1, 0.6]
    # Example: pointing straight up
    T_06[:3, :3] = rot_z(0) @ rot_y(0) @ rot_x(0)

    try:
        thetas = analytical_ik(T_06)
        print("Analytical IK solution (radians):")
        print(np.round(thetas, 4))
    except ValueError as e:
        print("IK Error:", e)


IK Error: Target unreachable: D > 1


When the last three axes intersect, the origins of link frames (4), {5}, and {6} are all located at this point of intersection. This point is given in base coordinates as